# PRD 2: Feature Engineering

This notebook designs and implements temporal and domain-inspired features to capture engine degradation patterns better than raw sensor readings. We ensure no data leakage by calculating features strictly up to the current cycle.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('deep')
import warnings
warnings.filterwarnings('ignore')



## 1. Load Processed Baseline Datasets


In [ ]:
DATA_PATH = os.path.join('..', 'data', 'processed')
train_df = pd.read_csv(os.path.join(DATA_PATH, 'train_processed.csv'))
val_df = pd.read_csv(os.path.join(DATA_PATH, 'validation_processed.csv'))
test_df = pd.read_csv(os.path.join(DATA_PATH, 'test_processed.csv'))

print(f"Train Shape: {train_df.shape}")
print(f"Val Shape: {val_df.shape}")
print(f"Test Shape: {test_df.shape}")



## 2. Identify Top Informative Sensors

Rather than engineering features for all sensors and cluttering the dataset, we identify the top 5 most informative sensors (highest linear correlation with RUL) and focus our temporal features on them.


In [ ]:
sensor_cols = [col for col in train_df.columns if 'sensor' in col]

# Calculate correlation with RUL
corrs = train_df[sensor_cols + ['RUL']].corr()['RUL'].abs().sort_values(ascending=False)
top_sensors = corrs.index[1:6].tolist() # Exclude RUL itself

print("Top 5 Informative Sensors chosen for Feature Engineering:")
print(top_sensors)



## 3. Temporal Feature Engineering

We generate the following features for the top sensors, strictly grouped by `engine_id` to prevent leakage:
- **Rolling Mean (w=15):** Smooths noise.
- **Rolling Std (w=15):** Captures increasing variance.
- **EMA (span=15):** Emphasizes recent trends.
- **Diff (shift=1):** Captures cycle-to-cycle changes.


In [ ]:
def engineer_features(df, sensors, window=15):
    df_engineered = df.copy()
    
    # Engine Lifetime Features
    df_engineered['log_cycle'] = np.log1p(df_engineered['cycle'])
    
    # Temporal Features
    grouped = df_engineered.groupby('engine_id')
    
    for sensor in sensors:
        # Rolling Mean
        df_engineered[f'{sensor}_roll_mean'] = grouped[sensor].transform(lambda x: x.rolling(window, min_periods=1).mean())
        
        # Rolling Std (fills NaN at cycle 1 with 0)
        df_engineered[f'{sensor}_roll_std'] = grouped[sensor].transform(lambda x: x.rolling(window, min_periods=1).std()).fillna(0)
        
        # EMA
        df_engineered[f'{sensor}_ema'] = grouped[sensor].transform(lambda x: x.ewm(span=window, min_periods=1).mean())
        
        # Difference from previous cycle
        df_engineered[f'{sensor}_diff'] = grouped[sensor].transform(lambda x: x.diff()).fillna(0)
        
    return df_engineered

train_eng = engineer_features(train_df, top_sensors, window=15)
val_eng = engineer_features(val_df, top_sensors, window=15)
test_eng = engineer_features(test_df, top_sensors, window=15)

print(f"Engineered Train Shape: {train_eng.shape}")
print(f"Total new features added: {train_eng.shape[1] - train_df.shape[1]}")



## 4. Feature Comparison (Baseline vs Engineered)

We train a baseline `RandomForestRegressor` on both datasets and compare performance on the validation set.


In [ ]:
# Define feature sets
drop_cols = ['engine_id', 'RUL'] # Do not include target or ID in features
raw_features = [col for col in train_df.columns if col not in drop_cols]
eng_features = [col for col in train_eng.columns if col not in drop_cols]

X_train_raw, y_train = train_df[raw_features], train_df['RUL']
X_val_raw, y_val = val_df[raw_features], val_df['RUL']

X_train_eng = train_eng[eng_features]
X_val_eng = val_eng[eng_features]

print("Training Baseline Model (Raw Features)...")
rf_raw = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_raw.fit(X_train_raw, y_train)
pred_raw = rf_raw.predict(X_val_raw)

print("Training Engineered Model (Raw + Engineered Features)...")
rf_eng = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_eng.fit(X_train_eng, y_train)
pred_eng = rf_eng.predict(X_val_eng)

def evaluate(y_true, y_pred, model_name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    print(f"{model_name} - RMSE: {rmse:.2f} | MAE: {mae:.2f}")

evaluate(y_val, pred_raw, "Raw Baseline Model")
evaluate(y_val, pred_eng, "Engineered Model")



## 5. Feature Importance Analysis

We analyze the top 15 most important features from the Engineered Model to determine if our designed features actually provide value to the model.


In [ ]:
importances = rf_eng.feature_importances_
feat_imp = pd.Series(importances, index=eng_features).sort_values(ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(x=feat_imp.head(15).values, y=feat_imp.head(15).index, palette='viridis')
plt.title('Top 15 Feature Importances (Engineered Model)')
plt.xlabel('Random Forest Importance')
plt.ylabel('Feature')
plt.show()

print("Observation: If engineered features (like roll_mean or ema) appear in the top 10, it confirms they successfully capture the degradation signal better than instantaneous raw values.")



## 6. Final Conclusion

- Temporal feature engineering was explored (Rolling statistics, EMA, Rate-of-Change, etc.).
- These engineered features produced **negligible improvement** over the baseline Random Forest model.
- Following the principle of keeping the model as simple as possible while maintaining performance, we will **discard these engineered features**.
- The final modeling phase will use the original processed feature set from PRD 1.
